# Análisis de resultados

Carga las corridas (de Drive o de `resultados.zip`) y corre `python -m analysis.make_all`, que genera las figuras (`docs/figures/`) y las tablas (`analysis/results/`) de los experimentos que ya tienen datos; los que faltan se saltan. No necesita SUMO: solo lee los CSV de `runs/`. En la Mac se hace igual: descomprime el zip en la raíz del repo y corre `python -m analysis.make_all`.

La demora total (`total_delay`) y el backlog van junto a la espera media en todas las tablas y figuras.

In [ ]:
import base64, getpass, os, subprocess
BRANCH = "main" #@param {type:"string"}
REPO_URL = "https://github.com/Alexistodj124/ProyectoFinalAR.git"
REPO_DIR = "/content/ProyectoFinalAR"
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = getpass.getpass("GITHUB_TOKEN: ")
AUTH = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()

def git(*args):
    r = subprocess.run(["git", "-c", f"http.extraheader=AUTHORIZATION: basic {AUTH}", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(AUTH, "***").replace(TOKEN, "***").strip())
    r.check_returncode()

if not os.path.isdir(REPO_DIR + "/.git"):
    git("clone", "--branch", BRANCH, REPO_URL, REPO_DIR)
else:
    git("-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
os.chdir(REPO_DIR)
git("log", "-1", "--oneline")

In [ ]:
!pip install -q numpy pandas scipy matplotlib pyyaml
os.environ["PYTHONPATH"] = REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

## Cargar las corridas
Elige la fuente: **drive** (la misma carpeta que usó el cuaderno de entrenamiento) o **zip** (sube `resultados.zip` con el panel de archivos o `files.upload()`). `RUNS_NAME` es la carpeta dentro del repo: `runs` para la matriz principal o `runs_x400` para la extensión de la celda B; en ese caso las figuras y tablas van a carpetas con ese sufijo, para no pisar las principales.

In [ ]:
SOURCE = "drive" #@param ["drive", "zip"]
RUNS_NAME = "runs" #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/ProyectoFinalAR" #@param {type:"string"}
ZIP_PATH = "/content/resultados.zip" #@param {type:"string"}
FIG_DIR = "docs/figures" if RUNS_NAME == "runs" else f"docs/figures_{RUNS_NAME}"
TABLE_DIR = "analysis/results" if RUNS_NAME == "runs" else f"analysis/results_{RUNS_NAME}"
if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p {RUNS_NAME} && rsync -a --exclude 'tb/' {DRIVE_ROOT}/{RUNS_NAME}/ {RUNS_NAME}/
else:
    if not os.path.exists(ZIP_PATH):
        from google.colab import files
        up = files.upload(); ZIP_PATH = "/content/" + next(iter(up))
    !unzip -oq {ZIP_PATH} -d .

## Qué hay
Corridas terminadas por experimento (marca `DONE`) y trabajos fallidos del manifiesto.

In [ ]:
import glob, pandas as pd
done = sorted(glob.glob(f"{RUNS_NAME}/E*/**/DONE", recursive=True))
print(pd.Series([p.split("/")[1] for p in done]).value_counts().sort_index().to_string() if done else "sin corridas")
m = f"{RUNS_NAME}/manifest.csv"
if os.path.exists(m):
    df = pd.read_csv(m)
    print("fallidos:", df[df["status"] == "failed"][["exp", "name", "seed"]].values.tolist() or "ninguno")

## Generar figuras y tablas

In [ ]:
!python -m analysis.make_all --runs-dir {RUNS_NAME} --fig-dir {FIG_DIR} --table-dir {TABLE_DIR}

## Ver las figuras

In [ ]:
from IPython.display import Image, display
for p in sorted(glob.glob(f"{FIG_DIR}/*.png")):
    print(p); display(Image(p))

## Ver las tablas (Markdown)

In [ ]:
from IPython.display import Markdown, display
for p in sorted(glob.glob(f"{TABLE_DIR}/*.md")):
    display(Markdown(open(p).read()))

## Descargar figuras y tablas
Copia el contenido a `docs/figures/` y `analysis/results/` del repo local.

In [ ]:
!zip -qr analisis.zip {FIG_DIR} {TABLE_DIR}
from google.colab import files
files.download("analisis.zip")